In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Simulations generations

## Parameters

In [5]:
# Meta-parameters
MAIN_FOLDER_PATH = '/home/david/Documents/code/loop_drift_estimation_output'

# Simulation generation parameters

P_CW_REWARD = 0.8
P_CCW_REWARD = 0
P_CW_INIT_RANGE = np.linspace(0.01,0.99,100)
STEPS_NUMBER = 100
NOISE_AMPLITUDE = 0.1
DRIFT_INIT = 0
DRIFT_VALUES_ARR = np.linspace(0.005,0.1,5)

DEFAULT_ARGS_DICT = {'p_cw_reward': P_CW_REWARD, 
             'p_ccw_reward': P_CCW_REWARD, 
             'p_cw_init': 0.5, 
             'steps_number': STEPS_NUMBER, 
             'noise_amplitude': NOISE_AMPLITUDE, 
             'drift_matrix': np.array([[DRIFT_VALUES_ARR[0], -DRIFT_VALUES_ARR[0]],
                                       [0                  , 0                   ]]), 
             'drift_init': DRIFT_INIT}


SIMULATIONS_SET_SIZE = 50
N_SETS = 100
SIMULATIONS_SET_SIZE_LIST = [SIMULATIONS_SET_SIZE]*N_SETS

# Fit parameters
STATES_NUMBER_RANGE = np.arange(2,12)
N_FITS = 50
N_JOBS = 5

# Plot HMM fitting result

In [81]:
example_set = 0
simu_index = 3

stacked_proba_sequences = []

fig = plt.figure(figsize=(7, 2.5), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.3, height_ratios=(1,0.3))
ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[1,0])

for simu_index in range(0,50):

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/simulation_set_{example_set}.pkl', 'rb') as file:
        simulations_set = pickle.load(file)

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][1]
    choices_sequence = [synth_data['choices'] for synth_data in simulations_set][simu_index]
    rewards_sequence = [synth_data['rewards'] for synth_data in simulations_set][simu_index]
    proba_sequence = [synth_data['p_cw'] for synth_data in simulations_set][simu_index]

    emissionmat = model.emissionprob_
    startprob = model.startprob_

    predicted_states_sequence = model.predict(np.reshape(choices_sequence,(-1,1)))
    predicted_proba_sequence = np.array([emissionmat[s,1] for s in predicted_states_sequence])
    colors = ('indigo','violet')
    colors_sequence = [colors[r] for r in rewards_sequence]

    ###

    steps = np.arange(len(proba_sequence))


    # ax1.scatter(steps, proba_sequence, marker='+', c=colors_sequence, s=70, linewidth=1.5, alpha=0.7)
    # ax1.plot(steps, proba_sequence, color='k', alpha=0.5)

    # ax1.scatter(steps, predicted_proba_sequence, marker='+', c='green', s=70, linewidth=1.5, alpha=0.5)
    # for s, p in enumerate(emissionmat[:,1]):
    #     ax1.axhline(p, linestyle='--', color='grey', linewidth=0.5, zorder=0)
    #     ax1.text(-1.7, p+0.05, rf'$P_{{s_{s}}}$', fontsize=7)

    # ax1.set_yticks(np.arange(0,1.1,0.2))
    # ax.set_ylim((0,0.5))
    # ax1.set_ylabel(r'$P_n$')
    # ax1.set_ylim(-0.1,1.1)

    ###


    # ax2.scatter(steps, choices_sequence, marker='+', c=colors_sequence, linewidth=1.5)
    # ax2.set_yticks((0,1))
    # ax2.set_yticklabels(('CCW', 'CW'))
    # ax2.set_ylim(-0.2,1.2)
    # ax2.set_xlabel('Step')

    stacked_proba_sequences.append(proba_sequence)

ax1.plot(steps, np.mean(stacked_proba_sequences,axis=0), color='k', alpha=1)
ax1.set_ylim(-0.1,1.1)


(-0.1, 1.1)

# Drift Estimation

## Generating "theoretical" average probability sequences

In [11]:
DRIFT_RANGE = np.linspace(0.001,0.11,200)

generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    temp_args_dict = copy.deepcopy(DEFAULT_ARGS_DICT)
        
    args = [temp_args_dict] + [5000] + [P_CW_INIT_RANGE]
    DRIFT_RANGE = np.linspace(0.001,0.11,200)

    test_average_probability_sequences_list = generate_test_average_probability_sequences_identical_drifts_random_init(DRIFT_RANGE, args)

    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences.pkl', 'wb') as file:
        pickle.dump([DRIFT_RANGE, test_average_probability_sequences_list], file)

else:
   
    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences.pkl', 'rb') as file:
        DRIFT_RANGE, test_average_probability_sequences_list = pickle.load(file)
test_average_probability_sequences_list = [test_seq[:STEPS_NUMBER] for test_seq in test_average_probability_sequences_list]
print(len(test_average_probability_sequences_list[0]))

100


## Minimizing mean square error

In [12]:
def estimate_drift(simulations_set, model, test_average_probability_sequences_list, drift_range):

    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
        
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    min_mse_index = np.where(mse_list==min_mse)[0]
    recovered_drift = drift_range[min_mse_index]

    return recovered_drift, min_mse_index

In [34]:
def estimate_drift(simulations_set, model, test_average_probability_sequences_list, drift_range):

    emissionmat = model.emissionprob_
    transmat = model.transmat_
    initial_state_distri = model.startprob_ # compute_initial_state_list_distri(model,test_data)

    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]
    
    mat_i = copy.deepcopy(transmat)

    average_proba_sequence_hmm = []

    for i in range(len(choices_sequences_list[0])):

        mat_i = np.matmul(mat_i,transmat)
        res = np.matmul(initial_state_distri,mat_i)*emissionmat[:,1]
    
        average_proba_sequence_hmm.append(np.sum(res))

    mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
        
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    min_mse_index = np.where(mse_list==min_mse)[0]
    recovered_drift = drift_range[min_mse_index]

    return recovered_drift, min_mse_index

In [13]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

for i in range(N_SETS):

    with open(f'{MAIN_FOLDER_PATH}/drift_{4}/simulation_set_{i}.pkl', 'rb') as file:
        simulations_set = pickle.load(file)

    with open(f'{MAIN_FOLDER_PATH}/drift_{4}/hmm_fit_output_{i}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][9]
    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    temp_mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
            
        temp_mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(temp_mse_list)
    min_mse_index = np.where(temp_mse_list==min_mse)[0]

    ax1.plot(DRIFT_RANGE, temp_mse_list, alpha=0.1)
    ax1.scatter(DRIFT_RANGE[min_mse_index], min_mse, alpha=0.1, c='k', marker='+')

In [14]:
recovered_drift_perdrift_list = []
N_STATES_INDEX = 3

for n in range(len(DRIFT_VALUES_ARR)):

    recovered_drift_list = []

    for i in range(N_SETS):

        ##############################
        ### Loading Data and Model ###
        ##############################

        with open(f'{MAIN_FOLDER_PATH}/drift_{n}/simulation_set_{i}.pkl', 'rb') as file:
            simulations_set = pickle.load(file)

        with open(f'{MAIN_FOLDER_PATH}/drift_{n}/hmm_fit_output_{i}.pkl', 'rb') as file:
            fit_output = pickle.load(file)

        model = fit_output['models'][N_STATES_INDEX]

        ####################
        ### Computations ###
        ####################

        recovered_drift, _ = estimate_drift(simulations_set, model, test_average_probability_sequences_list, DRIFT_RANGE)
        recovered_drift_list.append(recovered_drift[0])
        
        ##############################
        ### Saving Recovered drift ###
        ##############################
        
        # with open(f'{MAIN_FOLDER_PATH}/drift_{n}/recovered_drift_{i}.pkl', 'wb') as file:
        #     pickle.dump([test_average_probability_sequences_list,recovered_drift], file)
    recovered_drift_perdrift_list.append(recovered_drift_list)

In [35]:
stacked_recovered_drift_per_states_number = []

for n in range(len(DRIFT_VALUES_ARR)):

    recovered_drift_per_states_number = []
    
    for n_states_index in range(len(STATES_NUMBER_RANGE)):

        recovered_drift_per_states_number.append([])

        for i in range(N_SETS):

            average_proba_sequence_hmm = []

            ##############################
            ### Loading Data and Model ###
            ##############################

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/simulation_set_{i}.pkl', 'rb') as file:
                simulations_set = pickle.load(file)

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/hmm_fit_output_{i}.pkl', 'rb') as file:
                fit_output = pickle.load(file)

            # model = fit_output['models'][np.argmax(fit_output['scores'])]
            model = fit_output['models'][n_states_index]

            ####################
            ### Computations ###
            ####################

            test_data = [simulation['choices'] for simulation in simulations_set]
            recovered_drift, _ = estimate_drift(simulations_set, model, test_average_probability_sequences_list, DRIFT_RANGE)
            
            recovered_drift_per_states_number[-1].append(recovered_drift)
            
    stacked_recovered_drift_per_states_number.append(recovered_drift_per_states_number)


In [44]:
len(stacked_recovered_drift_per_states_number[0][0])

100

# Plots

In [ ]:
# example_set = 5

# average_proba_sequence_hmm = []

# ##############################
# ### Loading Data and Model ###
# ##############################

# with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
#     synthetic_data = pickle.load(file)

# with open(f'{simulations_folder_path}/n_{n_simulations}/model_score_fit_output_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
#     fit_output = pickle.load(file)

# model = fit_output['models'][np.argmax(fit_output['scores'])]

# test_data = [synth_data['choices'] for synth_data in synthetic_data]

# average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(test_data, model)

# mse_list = []

# # for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
# for test_average_probability_sequence in test_average_probability_sequences:
   
#     mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

# min_mse = np.min(mse_list)
# recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

# index_min_mse = np.where(mse_list==min_mse)[0][0]

# average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(test_data, model)

# ####################
# ### Computations ###
# ####################

# fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
# gs = fig.add_gridspec(1, 1)
# row = gs[:].subgridspec(1, 1, hspace=0.5)

# ax1 = plt.subplot(row[0,0])

# ax1.plot(average_proba_sequence_hmm)
# ax1.plot(test_average_probability_sequences[index_min_mse])
# ax1.text(20, 0.5, f'drift = {np.round(recovered_drift[0],4)}')

# ax1.set_ylim([0,1])
# ax1.set_xlabel('Step')
# ax1.set_ylabel('Average proba')

# ax1.legend()

/tmp/ipykernel_18712/2815055884.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()


In [42]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])


for i in range(len(STATES_NUMBER_RANGE)):

    temp_recovered_drift_list = stacked_recovered_drift_per_states_number[4][i]
    ax1.scatter(np.ones(N_SETS)*STATES_NUMBER_RANGE[i], temp_recovered_drift_list, marker='+', alpha=0.2, s=8)
    mean = np.mean(temp_recovered_drift_list)
    std = np.std(temp_recovered_drift_list)
    ax1.errorbar(STATES_NUMBER_RANGE[i],mean,std, marker='o', markersize=2, capsize=5)

ax1.axhline(DRIFT_VALUES_ARR[4], color='k', linestyle='--')

# ax1.set_xticks([])
ax1.set_title(f'{N_SETS} sets of {SIMULATIONS_SET_SIZE} simulations')

ax1.set_ylim([0,None])
ax1.set_xlabel('Number of states in HMM')
ax1.set_ylabel('Recovered drift')

ax1.legend()

/tmp/ipykernel_279625/3879313352.py:25: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()


In [48]:
len(np.ones(SIMULATIONS_SET_SIZE)*STATES_NUMBER_RANGE[i])
len(temp_recovered_drift_list)

100

In [34]:
DRIFT_VALUES_ARR

array([0.005  , 0.02875, 0.0525 , 0.07625, 0.1    ])

In [ ]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_perdrift_list[4], bins=np.linspace(0.01,0.2,101))
bin_width = histo[1][1] - histo[1][0]
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

ax1.axvline(0.050, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{N_SETS} sets of {SIMULATIONS_SET_SIZE} simulations')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Probability density')

ax1.legend()